# LeetCode 1311 — Get Watched Videos by Your Friends bằng Python

Link đề: https://leetcode.com/problems/get-watched-videos-by-your-friends/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Breadth-First Search, Graph Theory, Sorting

Skeleton LeetCode:

```python
class Solution:
    def watchedVideosByFriends(self, watchedVideos: list[list[str]], friends: list[list[int]], id: int, level: int) -> list[str]:
        ...
```

## 1. Mục tiêu học

1. Đọc đề đồ thị "mạng bạn bè": nhận ra **"level k" = những người có khoảng cách ngắn nhất đúng bằng k** ⇒ bài **BFS theo tầng**.
2. Viết BFS chuẩn với `visited` và biết **dừng đúng tầng `k`** (không lấy thừa tầng nông hơn hay sâu hơn).
3. Dùng `collections.Counter` để đếm tần suất, và `sorted(..., key=lambda v: (cnt[v], v))` để **sắp xếp theo nhiều tiêu chí**.
4. So sánh với baseline **Floyd–Warshall** (khoảng cách mọi cặp) và hiểu vì sao BFS là đủ với đồ thị không trọng số.
5. Tránh lỗi: quên đánh dấu chính mình, lấy "≤ k" thay vì "= k", đánh dấu `visited` muộn gây đếm trùng, sắp xếp thiếu tiêu chí phụ.

## 2. Đề bài & ví dụ

Có `n` người, mã số `0..n−1`. `watchedVideos[i]` là danh sách video người `i` đã xem, `friends[i]` là danh sách bạn của người `i` (quan hệ **hai chiều**).

- Video **level 1** = video mà **bạn** của bạn đã xem; **level 2** = video mà **bạn của bạn** đã xem; tổng quát, **level k** = video mà những người có **đường đi ngắn nhất tới bạn đúng bằng k** đã xem.

Cho `id` của bạn và `level`, trả về danh sách video level đó, sắp xếp theo **tần suất tăng dần**; cùng tần suất thì theo **thứ tự chữ cái tăng dần**.

**Ví dụ 1:** `watchedVideos = [["A","B"],["C"],["B","C"],["D"]]`, `friends = [[1,2],[0,3],[0,3],[1,2]]`, `id = 0`, `level = 1` → `["B","C"]` (bạn là 1, 2: B ×1, C ×2).

**Ví dụ 2:** cùng dữ liệu, `level = 2` → `["D"]` (người duy nhất cách 0 đúng 2 bước là 3).

```
    0 ---- 1
    |      |
    2 ---- 3
```

**Ràng buộc:** `2 <= n <= 100`; `1 <= watchedVideos[i].length <= 100`; `1 <= watchedVideos[i][j].length <= 8`; `0 <= friends[i].length < n`; `0 <= id < n`; `1 <= level < n`; nếu `friends[i]` chứa `j` thì `friends[j]` chứa `i`.

In [1]:
import random
import string
import time
from collections import Counter, deque
from typing import List

W = [["A", "B"], ["C"], ["B", "C"], ["D"]]
F = [[1, 2], [0, 3], [0, 3], [1, 2]]
examples = [
    ((W, F, 0, 1), ["B", "C"]),
    ((W, F, 0, 2), ["D"]),
]
for (w, f, i, k), want in examples:
    print(f"id={i}, level={k} -> mong đợi {want}")

id=0, level=1 -> mong đợi ['B', 'C']
id=0, level=2 -> mong đợi ['D']


## 3. Kiến thức nền

### 3.1 Đồ thị vô hướng dạng danh sách kề

`friends[i]` chính là **danh sách kề** của đỉnh `i`. Vì quan hệ hai chiều, đây là đồ thị **vô hướng, không trọng số**: mỗi cạnh dài 1.

In [2]:
for i, fr in enumerate(F):
    print(f"người {i}: bạn = {fr}, đã xem = {W[i]}")

người 0: bạn = [1, 2], đã xem = ['A', 'B']
người 1: bạn = [0, 3], đã xem = ['C']
người 2: bạn = [0, 3], đã xem = ['B', 'C']
người 3: bạn = [1, 2], đã xem = ['D']


### 3.2 BFS và khoảng cách ngắn nhất

Trên đồ thị không trọng số, BFS thăm các đỉnh theo **khoảng cách tăng dần**: tầng 0 là chính mình, tầng 1 là bạn, tầng 2 là bạn của bạn (chưa xuất hiện ở tầng trước)… Đỉnh được thăm lần đầu ở tầng `d` thì khoảng cách ngắn nhất đúng bằng `d`.

In [3]:
def bfs_dist(friends, src):
    dist = [-1] * len(friends)
    dist[src] = 0
    q = deque([src])
    while q:
        u = q.popleft()
        for v in friends[u]:
            if dist[v] == -1:
                dist[v] = dist[u] + 1
                q.append(v)
    return dist

print("khoảng cách từ 0:", bfs_dist(F, 0))

khoảng cách từ 0: [0, 1, 1, 2]


### 3.3 `Counter` và `update`

`Counter()` đếm tần suất; `cnt.update(list)` cộng thêm 1 cho mỗi phần tử trong list.

In [4]:
cnt = Counter()
cnt.update(["C"])
cnt.update(["B", "C"])
print(cnt, "| cnt['C'] =", cnt["C"], "| cnt['Z'] =", cnt["Z"], "(không có -> 0)")

Counter({'C': 2, 'B': 1}) | cnt['C'] = 2 | cnt['Z'] = 0 (không có -> 0)


### 3.4 Sắp xếp theo nhiều tiêu chí bằng `key` trả về tuple

Python so sánh tuple **từ trái sang phải**: `(1, "B") < (2, "A")` vì `1 < 2`; `(1, "B") < (1, "C")` vì hoà phần đầu, so phần sau. Vậy `key=lambda v: (cnt[v], v)` = "tần suất tăng, hoà thì chữ cái tăng".

In [5]:
cnt = Counter({"C": 2, "B": 1, "Z": 1, "A": 3})
print("chỉ theo tần suất  :", sorted(cnt, key=lambda v: cnt[v]), "(hoà B/Z giữ thứ tự chèn)")
print("(tần suất, tên)    :", sorted(cnt, key=lambda v: (cnt[v], v)))
print("so sánh tuple:", (1, "B") < (2, "A"), (1, "B") < (1, "C"))

chỉ theo tần suất  : ['B', 'Z', 'C', 'A'] (hoà B/Z giữ thứ tự chèn)
(tần suất, tên)    : ['B', 'Z', 'C', 'A']
so sánh tuple: True True


### 3.5 Sắp xếp ổn định (stable sort)

`sorted` của Python **ổn định**: phần tử bằng nhau theo khoá giữ nguyên thứ tự cũ. Nên có thể sắp hai lượt: **trước theo tiêu chí phụ (tên), sau theo tiêu chí chính (tần suất)** — kết quả giống dùng tuple.

In [6]:
items = list(cnt)
step1 = sorted(items)                       # theo tên
step2 = sorted(step1, key=lambda v: cnt[v]) # theo tần suất, ổn định
print(step1, "->", step2)

['A', 'B', 'C', 'Z'] -> ['B', 'Z', 'C', 'A']


## 4. Ý tưởng nền tảng / trực giác

Tính tay trên đồ thị ví dụ, `id = 0`:

| Tầng | Người (mới gặp) | Video của họ |
|---|---|---|
| 0 | {0} | A, B (của chính mình — không tính) |
| 1 | {1, 2} | C ; B, C |
| 2 | {3} | D |

- `level = 1`: đếm `B:1, C:2` ⇒ sắp theo (tần suất, tên): `["B", "C"]`.
- `level = 2`: người 3 cũng là bạn của 1 và 2 — nhưng **người 0 cũng là bạn của 1 và 2**! Nếu không nhớ đã thăm, tầng 2 sẽ gồm cả `0`, sai. Phải loại những người đã ở tầng nông hơn ⇒ tầng 2 = `{3}` ⇒ `["D"]`.

In [7]:
dist = bfs_dist(F, 0)
for k in (1, 2, 3):
    people = [p for p in range(len(F)) if dist[p] == k]
    cnt = Counter(v for p in people for v in W[p])
    print(f"level {k}: người {people}, đếm {dict(cnt)} -> {sorted(cnt, key=lambda v: (cnt[v], v))}")

level 1: người [1, 2], đếm {'C': 2, 'B': 1} -> ['B', 'C']
level 2: người [3], đếm {'D': 1} -> ['D']
level 3: người [], đếm {} -> []


## 5. Cách 1 — Brute Force / baseline: Floyd–Warshall

**Ý tưởng:** tính **khoảng cách ngắn nhất giữa mọi cặp** bằng Floyd–Warshall (ba vòng lặp lồng nhau), rồi lấy những người `p` có `dist[id][p] == level`, đếm video và sắp xếp. Không cần hiểu BFS, chỉ cần công thức `dist[i][j] = min(dist[i][j], dist[i][k] + dist[k][j])`.

In [8]:
def watched_floyd(watchedVideos, friends, id, level):
    n = len(friends)
    INF = float("inf")
    dist = [[INF] * n for _ in range(n)]
    for i in range(n):
        dist[i][i] = 0
        for j in friends[i]:
            dist[i][j] = 1
    for k in range(n):
        for i in range(n):
            dik = dist[i][k]
            if dik == INF:
                continue
            for j in range(n):
                if dik + dist[k][j] < dist[i][j]:
                    dist[i][j] = dik + dist[k][j]
    cnt = Counter()
    for p in range(n):
        if dist[id][p] == level:
            cnt.update(watchedVideos[p])
    return sorted(cnt, key=lambda v: (cnt[v], v))

for args, want in examples:
    print(watched_floyd(*args), "| mong đợi", want)

['B', 'C'] | mong đợi ['B', 'C']
['D'] | mong đợi ['D']


**Độ phức tạp:** gọi `n` = số người, `V` = tổng số video trong input, `U` = số video khác nhau ở level cần tìm.
- Time `O(n³ + V + U log U)`: với `n = 100` là `10^6` phép — chạy được, nhưng tính **mọi cặp** trong khi chỉ cần **một nguồn**.
- Space `O(n²)` cho ma trận khoảng cách.

**Nút thắt:** chỉ cần khoảng cách từ `id`, và đồ thị không trọng số ⇒ BFS một nguồn `O(n + E)` là đủ.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1:** đồ thị không trọng số + "đường đi ngắn nhất đúng bằng k" ⇒ **BFS từ `id`**, dừng sau `k` tầng; tầng cuối (frontier) chính là tập người cần lấy.

**Insight 2:** đánh dấu `visited` **ngay khi đưa vào hàng đợi/tầng mới** — mỗi người chỉ thuộc **một** tầng (tầng nhỏ nhất). Không đánh dấu ⇒ người ở tầng nông lọt vào tầng sâu (ví dụ chính `id` xuất hiện ở tầng 2).

**Insight 3:** đếm bằng `Counter`, sắp bằng khoá `(tần suất, tên)`.

**Quy tắc "nếu… thì…":**
- Nếu người `v` chưa thăm **thì** `v` thuộc tầng kế tiếp; đánh dấu ngay.
- Nếu đã thăm **thì** bỏ qua (khoảng cách của `v` đã nhỏ hơn hoặc bằng).
- Nếu frontier rỗng trước khi tới tầng `k` **thì** không ai ở tầng `k` ⇒ trả `[]`.

**Invariant:** sau `d` vòng mở rộng, `frontier` = đúng tập người có khoảng cách `d` tới `id`, `visited` = tập người có khoảng cách `<= d`.

In [9]:
# Minh hoạ invariant với đồ thị dài hơn: 0-1-2-3-4 và thêm cạnh 0-2
F2 = [[1, 2], [0, 2], [1, 3, 0], [2, 4], [3]]
visited, frontier = {0}, [0]
for d in range(1, 5):
    nxt = []
    for u in frontier:
        for v in F2[u]:
            if v not in visited:
                visited.add(v); nxt.append(v)
    frontier = nxt
    print(f"tầng {d}: frontier={frontier}, visited={sorted(visited)}")

tầng 1: frontier=[1, 2], visited=[0, 1, 2]
tầng 2: frontier=[3], visited=[0, 1, 2, 3]
tầng 3: frontier=[4], visited=[0, 1, 2, 3, 4]
tầng 4: frontier=[], visited=[0, 1, 2, 3, 4]


## 7. Các cách tiếp theo

### 7.1 Cách 2 — BFS với `deque` và mảng khoảng cách

BFS chuẩn, lưu `dist[v]`. Có thể **dừng sớm**: khi lấy ra đỉnh có `dist == level` thì không mở rộng tiếp (các đỉnh sâu hơn không cần).

In [10]:
def watched_bfs_deque(watchedVideos, friends, id, level):
    n = len(friends)
    dist = [-1] * n
    dist[id] = 0
    q = deque([id])
    cnt = Counter()
    while q:
        u = q.popleft()
        if dist[u] == level:
            cnt.update(watchedVideos[u])
            continue                         # không mở rộng quá tầng level
        for v in friends[u]:
            if dist[v] == -1:
                dist[v] = dist[u] + 1
                q.append(v)
    return sorted(cnt, key=lambda v: (cnt[v], v))

for args, want in examples:
    print(watched_bfs_deque(*args), "| mong đợi", want)

['B', 'C'] | mong đợi ['B', 'C']
['D'] | mong đợi ['D']


Code nộp LeetCode (cách 2):

In [11]:
class SolutionDeque:
    def watchedVideosByFriends(self, watchedVideos: List[List[str]], friends: List[List[int]], id: int, level: int) -> List[str]:
        dist = [-1] * len(friends)
        dist[id] = 0
        q = deque([id])
        cnt = Counter()
        while q:
            u = q.popleft()
            if dist[u] == level:
                cnt.update(watchedVideos[u])
                continue
            for v in friends[u]:
                if dist[v] == -1:
                    dist[v] = dist[u] + 1
                    q.append(v)
        return sorted(cnt, key=lambda v: (cnt[v], v))

print(SolutionDeque().watchedVideosByFriends(W, F, 0, 1))

['B', 'C']


**Độ phức tạp cách 2:** Time `O(n + E + V + U log U)` với `E` = tổng số quan hệ bạn bè; Space `O(n + U)`.

### 7.2 Cách 3 — BFS theo tầng bằng tập (frontier)

Viết gọn theo đúng định nghĩa "tầng": lặp `level` lần, mỗi lần dựng tầng kế tiếp từ tầng hiện tại, loại người đã thăm. Sau `level` vòng, `frontier` là tập người cần lấy.

In [12]:
def watched_frontier(watchedVideos, friends, id, level):
    visited = {id}
    frontier = {id}
    for _ in range(level):
        nxt = set()
        for u in frontier:
            for v in friends[u]:
                if v not in visited:
                    visited.add(v)
                    nxt.add(v)
        frontier = nxt
        if not frontier:                       # không còn ai ở xa hơn
            return []
    cnt = Counter(v for p in frontier for v in watchedVideos[p])
    return sorted(cnt, key=lambda v: (cnt[v], v))

for args, want in examples:
    print(watched_frontier(*args), "| mong đợi", want)

['B', 'C'] | mong đợi ['B', 'C']
['D'] | mong đợi ['D']


Code nộp LeetCode (cách 3):

In [13]:
class SolutionFrontier:
    def watchedVideosByFriends(self, watchedVideos: List[List[str]], friends: List[List[int]], id: int, level: int) -> List[str]:
        visited = {id}
        frontier = [id]
        for _ in range(level):
            nxt = []
            for u in frontier:
                for v in friends[u]:
                    if v not in visited:
                        visited.add(v)
                        nxt.append(v)
            frontier = nxt
        cnt = Counter()
        for p in frontier:
            cnt.update(watchedVideos[p])
        return sorted(cnt, key=lambda v: (cnt[v], v))

print(SolutionFrontier().watchedVideosByFriends(W, F, 0, 2))

['D']


**Độ phức tạp cách 3:** Time `O(n + E + V + U log U)`, Space `O(n + U)`. Chú ý: frontier rỗng thì `cnt` rỗng ⇒ trả `[]` tự nhiên (bản nộp không cần `return` sớm).

### 7.3 Cách 4 — BFS + sắp xếp ổn định hai lượt / gom nhóm theo tần suất

Phần sắp xếp có thể làm theo cách khác (hữu ích ở ngôn ngữ không có tuple-key, hoặc khi muốn "bucket"):
- Gom video theo tần suất: `buckets[f] = [các video có tần suất f]`.
- Duyệt `f` tăng dần, nối `sorted(buckets[f])`.

Kết quả giống hệt khoá `(cnt[v], v)`.

In [14]:
def watched_buckets(watchedVideos, friends, id, level):
    dist = bfs_dist(friends, id)
    cnt = Counter()
    for p, d in enumerate(dist):
        if d == level:
            cnt.update(watchedVideos[p])
    buckets = {}
    for v, f in cnt.items():
        buckets.setdefault(f, []).append(v)
    res = []
    for f in sorted(buckets):
        res.extend(sorted(buckets[f]))
    return res

for args, want in examples:
    print(watched_buckets(*args), "| mong đợi", want)

['B', 'C'] | mong đợi ['B', 'C']
['D'] | mong đợi ['D']


Code nộp LeetCode (cách 4):

In [15]:
class SolutionBuckets:
    def watchedVideosByFriends(self, watchedVideos: List[List[str]], friends: List[List[int]], id: int, level: int) -> List[str]:
        n = len(friends)
        dist = [-1] * n
        dist[id] = 0
        q = deque([id])
        while q:
            u = q.popleft()
            for v in friends[u]:
                if dist[v] == -1:
                    dist[v] = dist[u] + 1
                    q.append(v)
        cnt = Counter()
        for p in range(n):
            if dist[p] == level:
                cnt.update(watchedVideos[p])
        buckets = {}
        for v, f in cnt.items():
            buckets.setdefault(f, []).append(v)
        res = []
        for f in sorted(buckets):
            res.extend(sorted(buckets[f]))
        return res

print(SolutionBuckets().watchedVideosByFriends(W, F, 0, 1))

['B', 'C']


**Độ phức tạp cách 4:** như cách 2 (BFS toàn đồ thị, không dừng sớm), phần sắp xếp vẫn `O(U log U)` trong trường hợp xấu.

## 8. TRACE từng bước

Trace cách 3 trên đồ thị lớn hơn một chút (tự dựng để có nhiều tầng):

```
0 - 1 - 3 - 5
|   |
2 - 4
```
`watchedVideos = [["x"],["a","b"],["b"],["c","a"],["b","d"],["z"]]`, `id = 0`, `level = 2`.

| Vòng | frontier đầu vòng | Xét cạnh | nxt | visited sau |
|---|---|---|---|---|
| 1 | [0] | 0→1 mới, 0→2 mới | [1, 2] | {0,1,2} |
| 2 | [1, 2] | 1→0 cũ, 1→3 mới, 1→4 mới; 2→0 cũ, 2→4 cũ | [3, 4] | {0,1,2,3,4} |

Tầng 2 = `{3, 4}` ⇒ video: `c, a, b, d` mỗi cái 1 lần ⇒ `["a","b","c","d"]`.

In [16]:
W3 = [["x"], ["a", "b"], ["b"], ["c", "a"], ["b", "d"], ["z"]]
F3 = [[1, 2], [0, 3, 4], [0, 4], [1, 5], [1, 2], [3]]

def trace_frontier(watchedVideos, friends, id, level):
    visited, frontier = {id}, [id]
    for d in range(1, level + 1):
        nxt, notes = [], []
        for u in frontier:
            for v in friends[u]:
                if v not in visited:
                    visited.add(v); nxt.append(v); notes.append(f"{u}->{v} mới")
                else:
                    notes.append(f"{u}->{v} cũ")
        print(f"vòng {d}: frontier={frontier} | {', '.join(notes)} | nxt={nxt}")
        frontier = nxt
    cnt = Counter(v for p in frontier for v in watchedVideos[p])
    res = sorted(cnt, key=lambda v: (cnt[v], v))
    print("tầng", level, "=", frontier, "| đếm:", dict(cnt), "| kết quả:", res)
    return res

trace_frontier(W3, F3, 0, 2)

vòng 1: frontier=[0] | 0->1 mới, 0->2 mới | nxt=[1, 2]
vòng 2: frontier=[1, 2] | 1->0 cũ, 1->3 mới, 1->4 mới, 2->0 cũ, 2->4 cũ | nxt=[3, 4]
tầng 2 = [3, 4] | đếm: {'c': 1, 'a': 1, 'b': 1, 'd': 1} | kết quả: ['a', 'b', 'c', 'd']


['a', 'b', 'c', 'd']

In [17]:
trace_frontier(W3, F3, 0, 3)
print()
trace_frontier(W, F, 0, 2)

vòng 1: frontier=[0] | 0->1 mới, 0->2 mới | nxt=[1, 2]
vòng 2: frontier=[1, 2] | 1->0 cũ, 1->3 mới, 1->4 mới, 2->0 cũ, 2->4 cũ | nxt=[3, 4]
vòng 3: frontier=[3, 4] | 3->1 cũ, 3->5 mới, 4->1 cũ, 4->2 cũ | nxt=[5]
tầng 3 = [5] | đếm: {'z': 1} | kết quả: ['z']

vòng 1: frontier=[0] | 0->1 mới, 0->2 mới | nxt=[1, 2]
vòng 2: frontier=[1, 2] | 1->0 cũ, 1->3 mới, 2->0 cũ, 2->3 cũ | nxt=[3]
tầng 2 = [3] | đếm: {'D': 1} | kết quả: ['D']


['D']

## 9. Code nộp LeetCode

Bản chính thức: BFS theo tầng (cách 3) + `Counter` + sắp xếp khoá `(tần suất, tên)`.

In [18]:
class Solution:
    def watchedVideosByFriends(self, watchedVideos: list[list[str]], friends: list[list[int]], id: int, level: int) -> list[str]:
        visited = {id}
        frontier = [id]
        for _ in range(level):                 # mở rộng đúng `level` tầng
            nxt = []
            for u in frontier:
                for v in friends[u]:
                    if v not in visited:       # đánh dấu khi thêm vào tầng mới
                        visited.add(v)
                        nxt.append(v)
            frontier = nxt
        cnt = Counter()
        for p in frontier:                     # người có khoảng cách đúng bằng level
            cnt.update(watchedVideos[p])
        return sorted(cnt, key=lambda v: (cnt[v], v))

sol = Solution()
for args, want in examples:
    got = sol.watchedVideosByFriends(*args)
    print(got, "✔" if got == want else "✘")

['B', 'C'] ✔
['D'] ✔


## 10. Vì sao thuật toán đúng

**Bổ đề (invariant BFS theo tầng):** sau `d` vòng, `frontier = {p : dist(id, p) = d}` và `visited = {p : dist(id, p) <= d}`.

- `d = 0`: `frontier = visited = {id}` ✔.
- Giả sử đúng với `d`. Một người `v` được thêm vào `nxt` ⇔ `v` là bạn của ai đó trong `frontier` (khoảng cách `d`) và `v ∉ visited` (khoảng cách `> d`). Khi đó `dist(v) <= d + 1` và `> d` ⇒ `= d + 1`. Ngược lại, nếu `dist(v) = d + 1`, trên đường ngắn nhất, đỉnh liền trước `v` có khoảng cách `d` ⇒ thuộc `frontier` ⇒ `v` được xét và chưa thăm ⇒ được thêm. ✔

Do đó sau `level` vòng, `frontier` đúng là "người có đường đi ngắn nhất đúng bằng level". Đếm bằng `Counter` cộng mọi lần xuất hiện; `sorted` với khoá `(cnt, tên)` cho đúng thứ tự đề yêu cầu (so sánh tuple từ trái sang).

## 11. Phân tích Time / Space

Gọi `n` = số người (≤ 100), `E` = tổng độ dài các danh sách bạn (≤ `n²`), `V` = tổng số video của những người ở tầng `level`, `U` = số video khác nhau trong đó.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Floyd–Warshall | `O(n³ + V + U log U)` | `O(n²)` | baseline, mọi cặp |
| 2. BFS deque, dừng ở tầng | `O(n + E + V + U log U)` | `O(n + U)` | dừng sớm |
| 3. BFS theo tầng (frontier) | `O(n + E + V + U log U)` | `O(n + U)` | bản nộp chính |
| 4. BFS + bucket | `O(n + E + V + U log U)` | `O(n + U)` | sắp xếp kiểu gom nhóm |

Với ràng buộc của đề: `V <= 100 · 100 = 10^4`, so sánh chuỗi dài ≤ 8 ký tự ⇒ rất nhanh.

## 12. Bộ test

Ví dụ đề, edge case (không ai ở tầng `level` ⇒ `[]`, đồ thị không liên thông, người không có bạn, nhiều video trùng tên giữa nhiều người, hoà tần suất cần so tên), và đối chiếu mọi cách với Floyd–Warshall trên đồ thị ngẫu nhiên vô hướng.

In [19]:
ALL = [watched_floyd, watched_bfs_deque, watched_frontier, watched_buckets,
       Solution().watchedVideosByFriends, SolutionDeque().watchedVideosByFriends,
       SolutionFrontier().watchedVideosByFriends, SolutionBuckets().watchedVideosByFriends]
tests = [
    ((W, F, 0, 1), ["B", "C"]),
    ((W, F, 0, 2), ["D"]),
    ((W, F, 0, 3), []),                                        # không ai ở tầng 3
    ((W, F, 3, 1), ["B", "C"]),                                # bạn của 3 là 1, 2
    (([["a"], ["b"], ["c"]], [[1], [0], []], 0, 1), ["b"]),    # người 2 cô lập
    (([["a"], ["b"], ["c"]], [[1], [0], []], 2, 1), []),       # id không có bạn
    (([["x"], ["c", "b"]], [[1], [0]], 0, 1), ["b", "c"]),     # hoà -> theo tên
    (([["x"], ["b", "a", "b"], ["a", "c"]], [[1, 2], [0], [0]], 0, 1), ["c", "a", "b"]),
    ((W3, F3, 0, 2), ["a", "b", "c", "d"]),
    ((W3, F3, 0, 3), ["z"]),
]
for args, want in tests:
    for f in ALL:
        got = f(*args)
        assert got == want, (f, args, got, want)
print("Tất cả", len(tests), "test cố định qua cho", len(ALL), "hàm")

Tất cả 10 test cố định qua cho 8 hàm


In [20]:
def random_case(rng):
    n = rng.randint(2, 12)
    friends = [set() for _ in range(n)]
    for i in range(n):
        for j in range(i + 1, n):
            if rng.random() < 0.25:
                friends[i].add(j); friends[j].add(i)
    names = [c for c in "abcdef"]
    watched = [[rng.choice(names) for _ in range(rng.randint(1, 4))] for _ in range(n)]
    return watched, [sorted(s) for s in friends], rng.randrange(n), rng.randint(1, n - 1)

rng = random.Random(1311)
nonempty = 0
for _ in range(800):
    args = random_case(rng)
    want = watched_floyd(*args)
    nonempty += bool(want)
    for f in ALL[1:]:
        assert f(*args) == want, args
print("Đối chiếu 800 đồ thị ngẫu nhiên: khớp | số ca kết quả khác rỗng:", nonempty)

Đối chiếu 800 đồ thị ngẫu nhiên: khớp | số ca kết quả khác rỗng: 269


In [21]:
# Kích thước tối đa: n = 100, mỗi người 100 video, đồ thị dày
rng = random.Random(5)
n = 100
fr = [set() for _ in range(n)]
for i in range(n):
    for j in range(i + 1, n):
        if rng.random() < 0.05:
            fr[i].add(j); fr[j].add(i)
fr = [sorted(s) for s in fr]
wv = [["".join(rng.choice("abc") for _ in range(rng.randint(1, 3))) for _ in range(100)] for _ in range(n)]
t0 = time.perf_counter()
r1 = Solution().watchedVideosByFriends(wv, fr, 0, 2)
r2 = watched_floyd(wv, fr, 0, 2)
assert r1 == r2
print("n=100: khớp Floyd;", len(r1), "video;", f"{time.perf_counter() - t0:.3f}s")

n=100: khớp Floyd; 39 video; 0.044s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Không đánh dấu `visited` (hoặc quên đánh dấu chính `id`)

In [22]:
def wrong_no_visited(watchedVideos, friends, id, level):
    frontier = [id]
    for _ in range(level):
        frontier = list({v for u in frontier for v in friends[u]})   # SAI: không loại người đã gặp
    cnt = Counter(v for p in frontier for v in watchedVideos[p])
    return sorted(cnt, key=lambda v: (cnt[v], v))

print("sai :", wrong_no_visited(W, F, 0, 2))
print("đúng:", Solution().watchedVideosByFriends(W, F, 0, 2))

sai : ['A', 'B', 'D']
đúng: ['D']


**Vì sao sai:** "bạn của bạn" gồm cả chính người 0 (vì 1 và 2 đều là bạn của 0) ⇒ video `A, B` của chính mình lọt vào level 2. Đề yêu cầu khoảng cách **ngắn nhất** đúng bằng `level`. **Sửa:** giữ `visited` (khởi tạo `{id}`), chỉ thêm người chưa thăm.

### Lỗi 2 — Lấy mọi người có khoảng cách `<= level`

In [23]:
def wrong_le(watchedVideos, friends, id, level):
    dist = bfs_dist(friends, id)
    cnt = Counter()
    for p, d in enumerate(dist):
        if 1 <= d <= level:                                 # SAI: phải là d == level
            cnt.update(watchedVideos[p])
    return sorted(cnt, key=lambda v: (cnt[v], v))

print("sai :", wrong_le(W, F, 0, 2))
print("đúng:", Solution().watchedVideosByFriends(W, F, 0, 2))

sai : ['B', 'D', 'C']
đúng: ['D']


**Vì sao sai:** level 2 chỉ gồm người cách **đúng** 2 bước; bạn trực tiếp (1, 2) không thuộc level 2. **Sửa:** điều kiện `dist == level`, hoặc chỉ lấy `frontier` sau đúng `level` vòng.

### Lỗi 3 — Đánh dấu `visited` khi lấy ra thay vì khi đưa vào ⇒ đếm trùng

In [24]:
def wrong_late_mark(watchedVideos, friends, id, level):
    visited = set()
    frontier = [id]
    for _ in range(level):
        for u in frontier:
            visited.add(u)                                  # đánh dấu muộn
        nxt = []
        for u in frontier:
            for v in friends[u]:
                if v not in visited:
                    nxt.append(v)                           # SAI: v có thể được thêm nhiều lần
        frontier = nxt
    cnt = Counter(v for p in frontier for v in watchedVideos[p])
    return sorted(cnt, key=lambda v: (cnt[v], v))

# Hình thoi: 0-1, 0-2, 1-3, 2-3 và 1-4. Tầng 2 = {3, 4}
W4 = [["s"], ["p"], ["q"], ["x"], ["w", "z"]]
F4 = [[1, 2], [0, 3, 4], [0, 3], [1, 2], [1]]
print("sai :", wrong_late_mark(W4, F4, 0, 2))
print("đúng:", Solution().watchedVideosByFriends(W4, F4, 0, 2))

sai : ['w', 'z', 'x']
đúng: ['w', 'x', 'z']


**Vì sao sai:** người 3 là bạn của cả 1 và 2 nên bị đưa vào tầng 2 **hai lần** ⇒ video `x` bị đếm 2 lần, đẩy nó xuống cuối danh sách. **Sửa:** `visited.add(v)` **ngay khi** thêm `v` vào `nxt` (hoặc dùng `set` cho tầng mới).

### Lỗi 4 — Sắp xếp thiếu tiêu chí phụ (tên)

In [25]:
def wrong_sort(watchedVideos, friends, id, level):
    dist = bfs_dist(friends, id)
    cnt = Counter()
    for p, d in enumerate(dist):
        if d == level:
            cnt.update(watchedVideos[p])
    return sorted(cnt, key=lambda v: cnt[v])               # SAI: hoà giữ thứ tự chèn

args = ([["x"], ["c", "b"]], [[1], [0]], 0, 1)
print("sai :", wrong_sort(*args))
print("đúng:", Solution().watchedVideosByFriends(*args))

sai : ['c', 'b']
đúng: ['b', 'c']


**Vì sao sai:** `c` và `b` cùng tần suất 1; sắp ổn định giữ thứ tự xuất hiện `c, b`. Đề yêu cầu hoà thì theo chữ cái. **Sửa:** `key=lambda v: (cnt[v], v)`. (Ngược lại, chỉ sắp theo tên — `sorted(cnt)` — cũng sai vì bỏ qua tần suất.)

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Khoảng cách ngắn nhất đúng bằng k"** trên đồ thị **không trọng số** ⇒ BFS theo tầng, lặp đúng `k` vòng.
- **Luôn** `visited = {nguồn}` trước khi bắt đầu; đánh dấu khi **thêm** vào tầng mới.
- **"Sắp theo A tăng, hoà thì B tăng"** ⇒ `key=lambda x: (A, B)`; "A giảm" ⇒ `(-A, B)` khi A là số.
- **Đếm tần suất** ⇒ `Counter` + `update`.
- `n <= 100` ⇒ cả Floyd–Warshall `O(n³)` cũng chạy được — tốt để kiểm chứng.

## 15. Biến thể của bài

1. **Video của mọi người trong khoảng tầng `[lo, hi]`** (bài tập 5).
2. **Trả về video theo tần suất giảm dần, hoà theo tên giảm dần** ⇒ `sorted(cnt, key=lambda v: (cnt[v], v), reverse=True)`.
3. **Top-k video phổ biến nhất ở level k** ⇒ `heapq.nsmallest(k, cnt, key=lambda v: (-cnt[v], v))`.
4. **Đồ thị có trọng số** (độ thân thiết) ⇒ thay BFS bằng Dijkstra.
5. **LeetCode 863 — All Nodes Distance K in Binary Tree**: cùng ý tưởng "tầng k" trên cây (coi cây như đồ thị vô hướng).
6. **LeetCode 1162 / 542** — BFS đa nguồn theo tầng.

In [26]:
# Biến thể 2 và 3
import heapq
dist = bfs_dist(F3, 0)
cnt = Counter(v for p, d in enumerate(dist) if d in (1, 2) for v in W3[p])
print("đếm tầng 1..2:", dict(cnt))
print("giảm dần (tần suất, tên):", sorted(cnt, key=lambda v: (cnt[v], v), reverse=True))
print("top-2 phổ biến nhất:", heapq.nsmallest(2, cnt, key=lambda v: (-cnt[v], v)))

đếm tầng 1..2: {'a': 2, 'b': 3, 'c': 1, 'd': 1}
giảm dần (tần suất, tên): ['b', 'a', 'd', 'c']
top-2 phổ biến nhất: ['b', 'a']


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Floyd–Warshall | Khoảng cách mọi cặp | `O(n³)` + sort | `O(n²)` | `n` nhỏ, kiểm chứng |
| 2. BFS deque + dist | Dừng mở rộng ở tầng `level` | `O(n + E)` + sort | `O(n)` | Khi đã quen BFS chuẩn |
| 3. BFS theo tầng | Lặp `level` vòng trên frontier | `O(n + E)` + sort | `O(n)` | Bản nộp chuẩn, sát định nghĩa |
| 4. BFS + bucket | Gom theo tần suất rồi sort tên | `O(n + E)` + sort | `O(n + U)` | Ngôn ngữ không có tuple-key |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** Với `W, F` của đề: `Solution().watchedVideosByFriends(W, F, 3, 1)` và `(W, F, 1, 2)` trả về gì?

**Bài 2 (tính tay):** Với đồ thị `W3, F3` ở mục 8 và `id = 5`, liệt kê các tầng 1, 2, 3 và video ở tầng 3.

**Bài 3 (điền TODO):** Hoàn thành `people_at_level(friends, id, level)` trả về **danh sách đã sắp xếp** các người ở đúng tầng `level`.

**Bài 4 (tự viết lại):** Viết `sort_videos(cnt)` nhận `Counter` và trả danh sách theo (tần suất tăng, tên tăng) **không dùng tuple-key** (gợi ý: sắp xếp ổn định hai lượt).

**Bài 5 (mini challenge):** Viết `videos_in_range(watchedVideos, friends, id, lo, hi)`: video của những người có khoảng cách ngắn nhất nằm trong `[lo, hi]` (`1 <= lo <= hi`), cùng luật sắp xếp.

In [27]:
# Bài 3, 4, 5 — khung để tự điền (KHÔNG gọi khi chạy notebook)
def people_at_level_todo(friends, id, level):
    # TODO: BFS theo tầng, trả sorted(frontier)
    pass

def sort_videos_todo(cnt):
    # TODO: sorted theo tên, rồi sorted (ổn định) theo tần suất
    pass

def videos_in_range_todo(watchedVideos, friends, id, lo, hi):
    # TODO: BFS lấy dist, đếm video của người có lo <= dist <= hi
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1:** (1) Bạn của 3 là 1 và 2. (2) Từ 1: tầng 1 = {0, 3}, tầng 2 = {2}. (3) Người 2 xem `B, C`.

**Bài 2:** (1) Bạn của 5 chỉ có 3. (2) Tầng 2 = {1}. (3) Tầng 3 = {0, 4} ⇒ video `x, b, d`.

**Bài 3:** (1) `visited = {id}`, `frontier = [id]`. (2) Lặp `level` lần. (3) Trả `sorted(frontier)`.

**Bài 4:** (1) `step1 = sorted(cnt)` (theo tên). (2) `sorted(step1, key=cnt.get)` — ổn định nên giữ thứ tự tên khi hoà. (3) So kết quả với tuple-key để kiểm tra.

**Bài 5:** (1) Dùng `bfs_dist`. (2) Điều kiện `lo <= d <= hi` (bỏ qua `d == -1` là không tới được). (3) Sắp xếp như cũ.

## 19. Đáp án tham khảo cho bài tập

In [28]:
# Bài 1
assert Solution().watchedVideosByFriends(W, F, 3, 1) == ["B", "C"]
assert Solution().watchedVideosByFriends(W, F, 1, 2) == ["B", "C"]
# Bài 2
print("khoảng cách từ 5:", bfs_dist(F3, 5))
assert Solution().watchedVideosByFriends(W3, F3, 5, 3) == ["b", "d", "x"]

# Bài 3
def people_at_level(friends, id, level):
    visited, frontier = {id}, [id]
    for _ in range(level):
        nxt = []
        for u in frontier:
            for v in friends[u]:
                if v not in visited:
                    visited.add(v); nxt.append(v)
        frontier = nxt
    return sorted(frontier)

assert people_at_level(F, 0, 1) == [1, 2]
assert people_at_level(F3, 0, 2) == [3, 4]
assert people_at_level(F3, 0, 4) == []
print("Bài 1–3: OK")

khoảng cách từ 5: [3, 2, 4, 1, 3, 0]
Bài 1–3: OK


In [29]:
# Bài 4
def sort_videos(cnt):
    step1 = sorted(cnt)
    return sorted(step1, key=lambda v: cnt[v])

# Bài 5
def videos_in_range(watchedVideos, friends, id, lo, hi):
    dist = bfs_dist(friends, id)
    cnt = Counter()
    for p, d in enumerate(dist):
        if lo <= d <= hi:
            cnt.update(watchedVideos[p])
    return sorted(cnt, key=lambda v: (cnt[v], v))

rng = random.Random(99)
for _ in range(500):
    w, f, i, k = random_case(rng)
    lo = rng.randint(1, len(f) - 1); hi = rng.randint(lo, len(f) - 1)
    # kiểm tra bài 5 bằng cách gộp Counter của từng tầng lo..hi
    total = Counter()
    for lv in range(lo, hi + 1):
        n = len(f)
        d = bfs_dist(f, i)
        for p in range(n):
            if d[p] == lv:
                total.update(w[p])
    assert videos_in_range(w, f, i, lo, hi) == sorted(total, key=lambda v: (total[v], v))
    assert videos_in_range(w, f, i, k, k) == watched_floyd(w, f, i, k)
    c = Counter(rng.choice("abcde") for _ in range(rng.randint(0, 10)))
    assert sort_videos(c) == sorted(c, key=lambda v: (c[v], v))
print("Bài 4–5: OK — mọi assert đều qua")

Bài 4–5: OK — mọi assert đều qua


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "những đỉnh cách nguồn đúng k bước":**

```python
visited, frontier = {src}, [src]
for _ in range(k):
    nxt = []
    for u in frontier:
        for v in adj[u]:
            if v not in visited:
                visited.add(v)        # đánh dấu khi THÊM
                nxt.append(v)
    frontier = nxt
# frontier = các đỉnh có khoảng cách đúng bằng k
```

**Template đếm + sắp xếp nhiều tiêu chí:**

```python
cnt = Counter()
for p in frontier:
    cnt.update(items[p])
return sorted(cnt, key=lambda x: (cnt[x], x))
```

**Checklist trước khi submit:**
- [ ] `visited` khởi tạo gồm chính `id`.
- [ ] Lấy người có khoảng cách **đúng bằng** `level` (không phải `<=`).
- [ ] Đánh dấu khi thêm vào tầng mới (không đếm trùng người).
- [ ] Sắp xếp `(tần suất tăng, tên tăng)`.
- [ ] Không ai ở tầng `level` ⇒ trả `[]`.
- [ ] Đúng chữ ký: `watchedVideosByFriends(self, watchedVideos: list[list[str]], friends: list[list[int]], id: int, level: int) -> list[str]`.